# LangChain Basics & Chains
### Module 5 — LLM Application Frameworks (part 1)

**Goal:** learn the LangChain building blocks the way Omar Santos teaches them — **chat models**, **prompt templates**, **output parsers**, and **LCEL** chains (the `|` pipe) — then compose **branching** and **parallel** chains for security tasks. In Module 4 you built RAG by hand; LangChain is the framework that wires these pieces together cleanly.

> **Sources:** Omar Santos — repo `part2_prompt_templates` & `part3_prompt_chaining`; book Ch. 4; LinkedIn course Sections 2–3.

> **Setup:** `pip install langchain langchain-openai`. Needs `OPENAI_API_KEY`. Every model call below is **guarded** — with no key, the cell prints the built prompt/chain instead of crashing, so you can still read and follow along. You can also point `ChatOpenAI` at a local server.

## 🛠️ Setup

In [ ]:
# pip install langchain langchain-openai python-dotenv
import os
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv())
except Exception:
    pass

HAS_KEY = bool(os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-4o-mini"          # Omar uses a similar small model; swap as you like
print("OPENAI_API_KEY found:" , HAS_KEY, "| model:", MODEL)
print("(No key? The cells still show the prompts/chains they build.)")

# 💬 1 — Chat Models: One Interface for Every Provider

- LangChain wraps providers (OpenAI, Anthropic, …) in one interface
- `ChatModel` takes a list of messages, returns a message
- `.invoke(...)` runs it; `.batch` / `.stream` also available
- Swap models by changing one line — the rest of your code stays

> ### 🎤 Instructor Script
>
> LangChain's first gift is a standard interface to every model provider. A chat model takes a list of messages — each with a role like system or human — and returns a message. You call it with invoke. The point is portability: you can swap OpenAI for Anthropic or a local model by changing one line, and every chain you build keeps working. Here we make one direct call so you see the raw model before we add templates and chains around it.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="You are a concise cybersecurity assistant."),
    HumanMessage(content="In one sentence, what is SQL injection?"),
]

if HAS_KEY:
    model = ChatOpenAI(model=MODEL, temperature=0)
    reply = model.invoke(messages)
    print(reply.content)
else:
    print("[no key] Would send these messages to the model:")
    for m in messages:
        print(" ", type(m).__name__, "->", m.content)

# 🧱 2 — Prompt Templates (reusable prompts)

- A `ChatPromptTemplate` is a prompt with `{placeholders}`
- Fill the blanks at runtime with `.invoke({...})`
- Build from a string, or from system + human messages
- Keeps prompts consistent and parameterized

> ### 🎤 Instructor Script
>
> Hard-coding prompts doesn't scale, so LangChain gives us prompt templates: a prompt with named blanks you fill in at runtime. You can build one from a single template string or from a list of system and human messages, which is the style Omar uses. This mirrors his part-2 examples: a system message sets the role — an ethical-hacker assistant — and the human message carries the parameters. The template itself is just structure; nothing is sent to the model until we invoke it, which is why this cell runs even without a key.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful ethical-hacking assistant."),
    ("human", "Give a {adjective} explanation of the {vulnerability} vulnerability."),
])

filled = prompt.invoke({"adjective": "beginner-friendly", "vulnerability": "XSS"})
print("The template filled in:\n")
for m in filled.to_messages():
    print(" ", m.type, "->", m.content)

# 🔗 3 — Output Parsers + Your First LCEL Chain

- An output parser turns the model's reply into what you want
- `StrOutputParser` → plain text (the most common)
- **LCEL**: pipe pieces together with `|`
- `prompt | model | parser` — the workhorse chain

> ### 🎤 Instructor Script
>
> Two more pieces. An output parser converts the model's message into a usable form — the simplest, StrOutputParser, just gives you the text. And LangChain Expression Language, LCEL, lets you connect components with the pipe symbol, exactly like a Unix pipeline. The classic chain is prompt, then model, then parser. Once built, you invoke the whole chain with your variables and get clean text out. This three-piece pattern — prompt, model, parser — is the backbone of almost everything in LangChain.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an ethical hacker providing guidance on {topic}."),
    ("human", "List {tip_count} practical tips, each one line."),
])

if HAS_KEY:
    model = ChatOpenAI(model=MODEL, temperature=0)
    chain = prompt | model | StrOutputParser()        # <- LCEL pipe
    answer = chain.invoke({"topic": "scanning a web application", "tip_count": 5})
    print(answer)
else:
    print("[no key] Built chain:  prompt | model | StrOutputParser()")
    print("Would run with:", {"topic": "scanning a web application", "tip_count": 5})

# 🔀 4 — Branching Chains (route by condition)

- Pick different follow-up prompts based on the input
- `RunnableBranch` chooses a branch when a condition matches
- Example: route a vulnerability by **severity**
- Classify first, then respond differently per class

> ### 🎤 Instructor Script
>
> Real workflows aren't always a straight line. A branching chain looks at the input and routes to a different prompt depending on a condition — Omar's example routes a vulnerability by its severity to a critical, high, medium, or low response. RunnableBranch takes pairs of condition and chain, plus a default. Conceptually you first classify, then branch on the class. We build a small two-way branch — critical versus everything else — so the structure is clear; extending it to four levels is just more pairs.

In [ ]:
from langchain_core.runnables import RunnableBranch

if HAS_KEY:
    model = ChatOpenAI(model=MODEL, temperature=0)

    critical = ChatPromptTemplate.from_messages([
        ("system", "You are a senior incident responder."),
        ("human", "Give an URGENT mitigation plan for this CRITICAL issue: {vulnerability}"),
    ]) | model | StrOutputParser()

    routine = ChatPromptTemplate.from_messages([
        ("system", "You are a security analyst."),
        ("human", "Suggest standard remediation steps for this issue: {vulnerability}"),
    ]) | model | StrOutputParser()

    # route to "critical" when the text mentions critical/RCE, otherwise "routine"
    def is_critical(x):
        text = x["vulnerability"].lower()
        return ("critical" in text) or ("rce" in text)

    branch = RunnableBranch((is_critical, critical), routine)   # (condition, chain), default
    print(branch.invoke({"vulnerability": "Critical RCE in the web server"}))
else:
    print("[no key] Built a RunnableBranch: if text mentions 'critical'/'rce' -> urgent plan; else -> routine steps.")

# ⚡ 5 — Parallel Chains (do several at once)

- Run multiple prompts on the same input simultaneously
- `RunnableParallel` returns a dict of all results
- Example: recon **and** exploitation analysis together
- Then combine the parts into one report

> ### 🎤 Instructor Script
>
> Sometimes you want several independent analyses of the same input at once. A parallel chain runs multiple sub-chains on the same input and returns all their outputs in a dictionary. Omar's example analyzes a target for reconnaissance and for exploitation in parallel, then merges them into a single pen-test plan. We build a small version: two prompts about a target system run together, and a final step stitches the two answers into one. Parallel chains are how you keep complex workflows fast instead of waiting for each step in turn.

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnableLambda

if HAS_KEY:
    model = ChatOpenAI(model=MODEL, temperature=0)
    recon = ChatPromptTemplate.from_messages([
        ("system", "You are a recon specialist."),
        ("human", "List recon steps for the target: {target}"),
    ]) | model | StrOutputParser()
    exploit = ChatPromptTemplate.from_messages([
        ("system", "You are an exploitation specialist."),
        ("human", "List likely exploitation paths for the target: {target}"),
    ]) | model | StrOutputParser()

    both = RunnableParallel(recon=recon, exploitation=exploit)
    combine = RunnableLambda(lambda d: f"== RECON ==\n{d['recon']}\n\n== EXPLOITATION ==\n{d['exploitation']}")
    plan = both | combine
    print(plan.invoke({"target": "an Apache web server on port 80"}))
else:
    print("[no key] Built a RunnableParallel(recon=..., exploitation=...) then combined the two outputs.")

# ✅ Summary — LangChain Building Blocks

- **Chat models** — one interface for any provider
- **Prompt templates** — reusable, parameterized prompts
- **Output parsers** — turn replies into usable data
- **LCEL** (`|`) — compose prompt | model | parser
- **Branching & parallel** chains — real workflow logic
- Next: LangGraph (stateful agents) and RAG-with-LangChain

> ### 🎤 Instructor Script
>
> You now have LangChain's core vocabulary: chat models for provider-independent calls, prompt templates for reusable prompts, output parsers to shape the result, and LCEL to pipe them together with the bar symbol. On top of that, branching chains add conditional routing and parallel chains add concurrency — the logic real security workflows need. Next we'll meet LangGraph, which adds memory and loops for true agents, and then rebuild Module 4's RAG the LangChain way, which becomes your first mini-project.

In [ ]:
print("You can now build:  prompt | model | parser  — plus branching and parallel chains.")
print("Next notebooks: langgraph_agents.ipynb  and  langchain_rag.ipynb")